# Phase 1: Synthetic Data Generation


In [16]:
import numpy as np
import pandas as pd
from faker import Faker
from pathlib import Path
import random

# ---------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

# ---------------------------------------------------------------
# Project paths
# This works whether the notebook is opened from the project root
# or from the notebooks/ folder.
# ---------------------------------------------------------------
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data folder:", RAW_DATA_PATH.resolve())

Project root: C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook
Raw data folder: C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook\data\raw


## 1. Generate `customers_raw`

In [17]:
# ---------------------------------------------------------------
# 1. customers_raw
# ---------------------------------------------------------------
N_CUSTOMERS = 2000

segments = ["New", "Regular", "Premium", "Enterprise"]
segment_weights = [0.30, 0.40, 0.20, 0.10]

cities_states = [
    ("Mumbai", "Maharashtra"),
    ("Delhi", "Delhi"),
    ("Bengaluru", "Karnataka"),
    ("Hyderabad", "Telangana"),
    ("Chennai", "Tamil Nadu"),
    ("Pune", "Maharashtra"),
    ("Kolkata", "West Bengal"),
    ("Ahmedabad", "Gujarat"),
    ("Jaipur", "Rajasthan"),
    ("Lucknow", "Uttar Pradesh"),
]

customers = []

for cid in range(1, N_CUSTOMERS + 1):
    city, state = random.choice(cities_states)
    signup_date = fake.date_between(start_date="-2y", end_date="today")

    customers.append({
        "customer_id": cid,
        "customer_name": fake.name(),
        "customer_segment": np.random.choice(segments, p=segment_weights),
        "city": city,
        "state": state,
        "signup_date": signup_date,
    })

customers_df = pd.DataFrame(customers)
customers_df.to_csv(RAW_DATA_PATH / "customers_raw.csv", index=False)

print("customers_raw rows:", len(customers_df))
print("customers_raw saved:", (RAW_DATA_PATH / "customers_raw.csv").exists())
customers_df.head()

customers_raw rows: 2000
customers_raw saved: True


,customer_id,customer_name,customer_segment,city,state,signup_date
0,1,Patrick Sanchez,Regular,Delhi,Delhi,2025-11-30
1,2,Megan Mcclain,Enterprise,Mumbai,Maharashtra,2024-11-07
2,3,Amanda Davis,Premium,Chennai,Tamil Nadu,2024-09-27
3,4,Alyssa Gonzalez,Regular,Hyderabad,Telangana,2025-01-18
4,5,Jennifer Cole,New,Hyderabad,Telangana,2025-12-08


## 2. Generate `agents_raw`

In [18]:
# ---------------------------------------------------------------
# 2. agents_raw
# ---------------------------------------------------------------
N_AGENTS = 25

teams = ["General Support", "Refunds", "Payments", "Technical", "Delivery"]
shifts = ["Morning", "Evening", "Night"]
experience_levels = ["Junior", "Mid", "Senior"]
managers = [f"Manager_{i}" for i in range(1, 6)]

agents = []

for aid in range(1, N_AGENTS + 1):
    shift = random.choice(shifts)
    experience_level = random.choice(experience_levels)

    # Agents 3 and 7 are deliberately given lower stated capacity.
    # Later, they will receive a higher ticket load to create the overload pattern.
    if aid in (3, 7):
        max_daily_capacity = random.randint(15, 18)
    else:
        max_daily_capacity = random.randint(20, 28)

    agents.append({
        "agent_id": aid,
        "agent_name": fake.name(),
        "team": random.choice(teams),
        "shift": shift,
        "manager_name": random.choice(managers),
        "experience_level": experience_level,
        "max_daily_capacity": max_daily_capacity,
    })

agents_df = pd.DataFrame(agents)
agents_df.to_csv(RAW_DATA_PATH / "agents_raw.csv", index=False)

print("agents_raw rows:", len(agents_df))
print("agents_raw saved:", (RAW_DATA_PATH / "agents_raw.csv").exists())
agents_df.head()

agents_raw rows: 25
agents_raw saved: True


,agent_id,agent_name,team,shift,manager_name,experience_level,max_daily_capacity
0,1,Mark Mcintyre,Refunds,Night,Manager_5,Senior,24
1,2,Julian Willis,Technical,Evening,Manager_5,Senior,20
2,3,Phillip Pearson,Technical,Evening,Manager_2,Senior,17
3,4,Keith Griffin,Technical,Night,Manager_1,Mid,24
4,5,Kathryn Patton,Technical,Night,Manager_4,Junior,22


## 3. Generate `sla_policy_raw`

In [19]:
# ---------------------------------------------------------------
# 3. sla_policy_raw
# 4 priorities × 6 categories = 24 rows
# ---------------------------------------------------------------
priorities = ["Low", "Medium", "High", "Critical"]
categories = ["Refund", "Payment", "Delivery", "Login", "Account", "Product"]

# Base SLA hours by priority: first response SLA, resolution SLA
base_sla = {
    "Critical": (1, 4),
    "High": (2, 12),
    "Medium": (4, 24),
    "Low": (8, 48),
}

sla_rows = []

for priority in priorities:
    fr_base, res_base = base_sla[priority]

    for category in categories:
        # Refund cases are given slightly longer policy windows,
        # but they will still breach more often due to operational delay.
        if category == "Refund":
            fr_hours = round(fr_base * 1.5, 1)
            res_hours = round(res_base * 1.5, 1)
        else:
            fr_hours = fr_base
            res_hours = res_base

        sla_rows.append({
            "priority": priority,
            "category": category,
            "first_response_sla_hours": fr_hours,
            "resolution_sla_hours": res_hours,
        })

sla_policy_df = pd.DataFrame(sla_rows)
sla_policy_df.to_csv(RAW_DATA_PATH / "sla_policy_raw.csv", index=False)

print("sla_policy_raw rows:", len(sla_policy_df))
print("Duplicate priority-category pairs:", sla_policy_df.duplicated(subset=["priority", "category"]).sum())
print("sla_policy_raw saved:", (RAW_DATA_PATH / "sla_policy_raw.csv").exists())
sla_policy_df

sla_policy_raw rows: 24
Duplicate priority-category pairs: 0
sla_policy_raw saved: True


,priority,category,first_response_sla_hours,resolution_sla_hours
0,Low,Refund,12.0,72.0
1,Low,Payment,8.0,48.0
2,Low,Delivery,8.0,48.0
3,Low,Login,8.0,48.0
4,Low,Account,8.0,48.0
5,Low,Product,8.0,48.0
6,Medium,Refund,6.0,36.0
7,Medium,Payment,4.0,24.0
8,Medium,Delivery,4.0,24.0
9,Medium,Login,4.0,24.0


## 4. Generate `tickets_raw` before data quality injection

This cell creates the clean base ticket data and embeds the 4 planned business patterns


In [20]:
# ---------------------------------------------------------------
# 4. tickets_raw base data
# ---------------------------------------------------------------
N_TICKETS = 9000

channels = ["Email", "Chat", "Phone", "WhatsApp"]
statuses = ["Open", "Pending", "Escalated", "Resolved"]

# ---------------------------------------------------------------
# Agent assignment with overload pattern
# Agents 3 and 7 combined should get around 22% to 28% of all tickets.
# ---------------------------------------------------------------
overload_share = np.random.uniform(0.22, 0.28)
n_overload_tickets = int(N_TICKETS * overload_share)
n_normal_tickets = N_TICKETS - n_overload_tickets

overload_agent_ids = [3, 7]
normal_agent_ids = [a for a in range(1, N_AGENTS + 1) if a not in overload_agent_ids]

agent_assignments = (
    list(np.random.choice(overload_agent_ids, size=n_overload_tickets))
    + list(np.random.choice(normal_agent_ids, size=n_normal_tickets))
)
np.random.shuffle(agent_assignments)

# Lookup dictionaries
agent_shift_map = dict(zip(agents_df["agent_id"], agents_df["shift"]))

sla_lookup = {
    (row.priority, row.category): (
        row.first_response_sla_hours,
        row.resolution_sla_hours,
    )
    for row in sla_policy_df.itertuples()
}

# Category-level resolution breach target probabilities
category_breach_target = {
    "Refund": np.random.uniform(0.60, 0.65),
    "Payment": np.random.uniform(0.30, 0.40),
    "Delivery": np.random.uniform(0.25, 0.35),
    "Login": np.random.uniform(0.10, 0.20),
    "Account": np.random.uniform(0.15, 0.25),
    "Product": np.random.uniform(0.20, 0.30),
}

# Channel-level first response delay ranges (hours)
channel_fr_range = {
    "Chat": (1, 2),
    "Phone": (1, 3),
    "Email": (3, 5),
    "WhatsApp": (6, 9),
}

# Critical & High tickets receive faster first responses
priority_fr_multiplier = {
    "Critical": 0.55,
    "High": 0.75,
    "Medium": 1.00,
    "Low": 1.00,
}

# Shift-level breach boost
shift_breach_boost = {
    "Morning": 0.00,
    "Evening": 0.06,
    "Night": 0.25,
}

priority_weights = [0.35, 0.35, 0.20, 0.10]

tickets = []

start_date = pd.Timestamp("2026-01-01")
end_date = pd.Timestamp("2026-06-30")
date_range_seconds = int((end_date - start_date).total_seconds())

for i in range(N_TICKETS):

    ticket_id = i + 1
    customer_id = random.randint(1, N_CUSTOMERS)
    agent_id = int(agent_assignments[i])

    priority = np.random.choice(priorities, p=priority_weights)
    category = random.choice(categories)
    channel = random.choice(channels)

    shift = agent_shift_map[agent_id]
    first_response_sla, resolution_sla = sla_lookup[(priority, category)]

    # -----------------------------------------------------------
    # Ticket Status
    # -----------------------------------------------------------
    status = np.random.choice(statuses, p=[0.10, 0.10, 0.05, 0.75])

    # -----------------------------------------------------------
    # Ticket Creation Time
    # -----------------------------------------------------------
    if status == "Resolved":

        created_at = start_date + pd.Timedelta(
            seconds=random.randint(0, date_range_seconds)
        )

    else:

        recency_bucket = np.random.choice(
            ["recent", "aging", "stuck"],
            p=[0.70, 0.20, 0.10]
        )

        if recency_bucket == "recent":
            days_back = np.random.uniform(0, 3)
        elif recency_bucket == "aging":
            days_back = np.random.uniform(3, 14)
        else:
            days_back = np.random.uniform(14, 45)

        created_at = end_date - pd.Timedelta(days=days_back)

    # -----------------------------------------------------------
    # First Response Time
    # -----------------------------------------------------------
    fr_low, fr_high = channel_fr_range[channel]

    first_response_hours = (
        np.random.uniform(fr_low, fr_high)
        * priority_fr_multiplier[priority]
    )

    first_response_at = created_at + pd.Timedelta(
        hours=first_response_hours
    )

    # -----------------------------------------------------------
    # Resolution Breach Probability
    # -----------------------------------------------------------
    breach_prob = category_breach_target[category]

    if agent_id in overload_agent_ids:
        breach_prob = max(
            breach_prob,
            np.random.uniform(0.45, 0.55)
        )

    breach_prob = min(
        0.95,
        breach_prob + shift_breach_boost[shift]
    )

    will_breach = np.random.random() < breach_prob

    # -----------------------------------------------------------
    # Resolution Duration (measured from ticket creation)
    # -----------------------------------------------------------
    if will_breach:
        resolution_hours = (
            resolution_sla * np.random.uniform(1.1, 2.5)
        )
    else:
        resolution_hours = (
            resolution_sla * np.random.uniform(0.3, 0.95)
        )

    if agent_id in overload_agent_ids:
        resolution_hours *= np.random.uniform(1.1, 1.4)

    # -----------------------------------------------------------
    # Resolution Timestamp
    # -----------------------------------------------------------
    if status == "Resolved":

        resolved_at = created_at + pd.Timedelta(
            hours=resolution_hours
        )

        # A ticket cannot be resolved before first response
        if resolved_at <= first_response_at:

            resolved_at = first_response_at + pd.Timedelta(
                minutes=np.random.uniform(3, 20)
            )

        satisfaction_score = np.random.randint(1, 6)

        if will_breach and satisfaction_score > 2:

            satisfaction_score = np.random.choice(
                [1, 2, 3],
                p=[0.4, 0.4, 0.2]
            )

    else:

        resolved_at = pd.NaT
        satisfaction_score = np.nan

    # -----------------------------------------------------------
    # Save Ticket
    # -----------------------------------------------------------
    tickets.append({

        "ticket_id": ticket_id,
        "customer_id": customer_id,
        "created_at": created_at,
        "first_response_at": first_response_at,
        "resolved_at": resolved_at,
        "priority": priority,
        "category": category,
        "agent_id": agent_id,
        "channel": channel,
        "status": status,
        "satisfaction_score": satisfaction_score,

    })

tickets_df = pd.DataFrame(tickets)

print("tickets_df shape before DQ injection:", tickets_df.shape)
tickets_df.head()

tickets_df shape before DQ injection: (9000, 11)


,ticket_id,customer_id,created_at,first_response_at,resolved_at,priority,category,agent_id,channel,status,satisfaction_score
0,1,93,2026-05-19 13:34:44.000000000,2026-05-19 15:24:00.590638230,2026-05-20 16:34:33.220753724,Medium,Refund,12,Chat,Resolved,3.0
1,2,683,2026-01-29 22:06:49.000000000,2026-01-30 04:51:19.859427339,2026-01-30 20:07:56.778111013,Medium,Login,22,WhatsApp,Resolved,1.0
2,3,1242,2026-03-05 16:25:53.000000000,2026-03-05 18:10:09.205419314,2026-03-06 17:54:33.788049239,Low,Account,2,Chat,Resolved,5.0
3,4,1908,2026-06-26 20:39:44.471629423,2026-06-26 21:44:17.603120819,NaT,High,Account,15,Phone,Escalated,NaN
4,5,333,2026-04-26 06:57:26.000000000,2026-04-26 07:42:50.201143957,2026-04-26 16:03:17.800426286,High,Login,24,Phone,Resolved,3.0


## 5. Inject 5 data quality issues and save `tickets_raw.csv`

This cell intentionally injects the 5 planned data quality issues. It then saves `tickets_raw.csv` once, after the final raw ticket data is ready.


In [21]:
# =================================================================
# INJECT 5 DATA QUALITY ISSUES
# =================================================================

n = len(tickets_df)

# DQ001: Duplicate ticket_id (~1.5% -> ~135 duplicate rows)
n_dup = int(n * 0.015)
dup_rows = tickets_df.sample(n=n_dup, random_state=SEED).copy()
tickets_df = pd.concat([tickets_df, dup_rows], ignore_index=True)

# DQ002: Null agent_id (~2% -> ~180 rows)
n_null_agent = int(n * 0.02)
null_agent_idx = np.random.choice(
    tickets_df.index,
    size=n_null_agent,
    replace=False,
)
tickets_df.loc[null_agent_idx, "agent_id"] = np.nan

# DQ003: resolved_at before created_at (~1% -> ~90 rows)
n_bad_ts = int(n * 0.01)
resolved_mask = tickets_df["resolved_at"].notna()
eligible_idx = tickets_df[resolved_mask].index
bad_ts_idx = np.random.choice(
    eligible_idx,
    size=min(n_bad_ts, len(eligible_idx)),
    replace=False,
)
tickets_df.loc[bad_ts_idx, "resolved_at"] = (
    tickets_df.loc[bad_ts_idx, "created_at"] - pd.Timedelta(hours=5)
)

# DQ004: status = Resolved but resolved_at is NULL (~1.2% -> ~108 rows)
n_resolved_null = int(n * 0.012)
resolved_status_idx = tickets_df[tickets_df["status"] == "Resolved"].index
resolved_null_idx = np.random.choice(
    resolved_status_idx,
    size=min(n_resolved_null, len(resolved_status_idx)),
    replace=False,
)
tickets_df.loc[resolved_null_idx, "resolved_at"] = pd.NaT

# DQ005: satisfaction_score outside 1-5 (~1% -> ~90 rows)
n_bad_score = int(n * 0.01)
non_null_score_idx = tickets_df[tickets_df["satisfaction_score"].notna()].index
bad_score_idx = np.random.choice(
    non_null_score_idx,
    size=min(n_bad_score, len(non_null_score_idx)),
    replace=False,
)
tickets_df.loc[bad_score_idx, "satisfaction_score"] = np.random.choice(
    [0, 6, -1],
    size=len(bad_score_idx),
)

# Shuffle final raw ticket table
tickets_df = tickets_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Keep nullable integer style for clean CSV output
tickets_df["agent_id"] = tickets_df["agent_id"].astype("Int64")
tickets_df["satisfaction_score"] = tickets_df["satisfaction_score"].astype("Int64")

# Save final tickets_raw after DQ injection
tickets_df.to_csv(RAW_DATA_PATH / "tickets_raw.csv", index=False)

print("tickets_df shape after DQ injection:", tickets_df.shape)
print("tickets_raw saved:", (RAW_DATA_PATH / "tickets_raw.csv").exists())

tickets_df shape after DQ injection: (9135, 11)
tickets_raw saved: True


## 6. Sanity checks

These checks confirm that the injected data quality issues are present in the expected amounts.


In [22]:
print("\n--- SANITY CHECKS ---")

print("Total ticket rows:", len(tickets_df))
print("Duplicate ticket_ids:", tickets_df["ticket_id"].duplicated().sum())
print("Null agent_id:", tickets_df["agent_id"].isna().sum())

print(
    "resolved_at before created_at:",
    (
        tickets_df["resolved_at"].notna()
        & (tickets_df["resolved_at"] < tickets_df["created_at"])
    ).sum(),
)

print(
    "Resolved status with null resolved_at:",
    (
        (tickets_df["status"] == "Resolved")
        & (tickets_df["resolved_at"].isna())
    ).sum(),
)

print(
    "Invalid satisfaction scores:",
    tickets_df["satisfaction_score"].apply(
        lambda x: pd.notna(x) and (x < 1 or x > 5)
    ).sum(),
)

print("\nTickets per agent - top 5:")
print(tickets_df["agent_id"].value_counts().head(5))

print("\nCategory distribution:")
print(tickets_df["category"].value_counts())


--- SANITY CHECKS ---
Total ticket rows: 9135
Duplicate ticket_ids: 135
Null agent_id: 180
resolved_at before created_at: 88
Resolved status with null resolved_at: 108
Invalid satisfaction scores: 90

Tickets per agent - top 5:
agent_id
3     1088
7     1030
15     336
24     330
18     326
Name: count, dtype: Int64

Category distribution:
category
Login       1575
Product     1548
Payment     1537
Account     1524
Refund      1493
Delivery    1458
Name: count, dtype: int64


## 7. Prepare validation data

This validation dataframe is only for checking whether the planned business patterns are visible. It does not change the raw CSV files.


In [23]:
# ---------------------------------------------------------------
# Root-cause validation before SQL
# ---------------------------------------------------------------
validation_df = tickets_df.copy()

# Remove duplicate ticket rows for validation only
validation_df = validation_df.drop_duplicates(subset=["ticket_id"], keep="first")

# Remove missing agent_id rows for validation only
validation_df = validation_df[validation_df["agent_id"].notna()].copy()

# Remove invalid timestamp rows for validation only
validation_df = validation_df[
    ~(
        validation_df["resolved_at"].notna()
        & (validation_df["resolved_at"] < validation_df["created_at"])
    )
].copy()

# Join SLA policy
validation_df = validation_df.merge(
    sla_policy_df,
    on=["priority", "category"],
    how="left",
)

# Join agent attributes
validation_df = validation_df.merge(
    agents_df[["agent_id", "shift", "team", "max_daily_capacity"]],
    on="agent_id",
    how="left",
)

# Time metrics
validation_df["first_response_time_hrs"] = (
    validation_df["first_response_at"] - validation_df["created_at"]
).dt.total_seconds() / 3600

validation_df["resolution_time_hrs"] = (
    validation_df["resolved_at"] - validation_df["created_at"]
).dt.total_seconds() / 3600

# SLA flags
validation_df["first_response_breached"] = (
    validation_df["first_response_time_hrs"]
    > validation_df["first_response_sla_hours"]
)

validation_df["resolution_breached"] = (
    (validation_df["status"] == "Resolved")
    & validation_df["resolved_at"].notna()
    & (validation_df["resolution_time_hrs"] > validation_df["resolution_sla_hours"])
)

print("Validation dataframe shape:", validation_df.shape)

Validation dataframe shape: (8736, 20)


## 8. Validate root-cause pattern 1: Refund category breach

In [24]:
category_breach = (
    validation_df[validation_df["status"] == "Resolved"]
    .groupby("category")["resolution_breached"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
)

print("Resolution breach rate by category (%):")
print(category_breach)

Resolution breach rate by category (%):
category
Refund      68.14
Delivery    50.93
Payment     45.21
Product     41.67
Login       34.36
Account     34.30
Name: resolution_breached, dtype: float64


## 9. Validate root-cause pattern 2: WhatsApp first-response delay

In [25]:
channel_response = (
    validation_df
    .groupby("channel")["first_response_time_hrs"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("Average first response time by channel:")
print(channel_response)

Average first response time by channel:
channel
WhatsApp    6.81
Email       3.61
Phone       1.81
Chat        1.35
Name: first_response_time_hrs, dtype: float64


## 10. Validate root-cause pattern 3: Agent 3 and Agent 7 overload

In [26]:
agent_summary = (
    validation_df
    .groupby("agent_id")
    .agg(
        tickets_handled=("ticket_id", "count"),
        breach_rate=("resolution_breached", "mean"),
        avg_resolution_time_hrs=("resolution_time_hrs", "mean"),
        team=("team", "first"),
        shift=("shift", "first"),
        max_daily_capacity=("max_daily_capacity", "first"),
    )
    .sort_values("tickets_handled", ascending=False)
)

agent_summary["breach_rate"] = (agent_summary["breach_rate"] * 100).round(2)
agent_summary["avg_resolution_time_hrs"] = agent_summary["avg_resolution_time_hrs"].round(2)

print("Agent performance summary - top 10 by ticket count:")
print(agent_summary.head(10))

Agent performance summary - top 10 by ticket count:
          tickets_handled  breach_rate  avg_resolution_time_hrs       team  \
agent_id                                                                     
3                    1062        50.28                    50.29  Technical   
7                    1008        48.02                    50.13   Payments   
15                    329        27.05                    30.29   Delivery   
24                    324        24.38                    32.75    Refunds   
18                    317        22.71                    29.34    Refunds   
19                    317        41.96                    42.83   Payments   
25                    307        38.44                    34.51   Delivery   
12                    307        20.52                    29.99    Refunds   
20                    305        26.23                    31.72    Refunds   
17                    295        33.22                    36.50   Payments   

           

## 11. Validate root-cause pattern 4: Night shift delay

In [27]:
shift_summary = (
    validation_df[validation_df["status"] == "Resolved"]
    .groupby("shift")
    .agg(
        tickets=("ticket_id", "count"),
        breach_rate=("resolution_breached", "mean"),
        avg_resolution_time_hrs=("resolution_time_hrs", "mean"),
    )
)

shift_summary["breach_rate"] = (shift_summary["breach_rate"] * 100).round(2)
shift_summary["avg_resolution_time_hrs"] = shift_summary["avg_resolution_time_hrs"].round(2)

print("Shift breach summary:")
print(shift_summary.sort_values("breach_rate", ascending=False))

Shift breach summary:
         tickets  breach_rate  avg_resolution_time_hrs
shift                                                 
Night       1543        52.30                    37.57
Morning     1578        46.58                    39.62
Evening     3417        42.17                    36.08


## 12. Final file check

This cell only verifies that all required raw CSV files exist. It does not save or overwrite anything.


In [28]:
required_files = [
    "customers_raw.csv",
    "agents_raw.csv",
    "sla_policy_raw.csv",
    "tickets_raw.csv",
]

print("\n--- FINAL FILE CHECK ---")
for file_name in required_files:
    file_path = RAW_DATA_PATH / file_name
    print(f"{file_name}: exists={file_path.exists()} | path={file_path}")


--- FINAL FILE CHECK ---
customers_raw.csv: exists=True | path=C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook\data\raw\customers_raw.csv
agents_raw.csv: exists=True | path=C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook\data\raw\agents_raw.csv
sla_policy_raw.csv: exists=True | path=C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook\data\raw\sla_policy_raw.csv
tickets_raw.csv: exists=True | path=C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook\data\raw\tickets_raw.csv


In [29]:
print("Current working directory:", Path.cwd())
print("RAW_DATA_PATH:", RAW_DATA_PATH.resolve())
import os
print("Contents of RAW_DATA_PATH parent:", os.listdir(RAW_DATA_PATH.parent) if RAW_DATA_PATH.parent.exists() else "parent folder doesn't exist")

Current working directory: C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook
RAW_DATA_PATH: C:\Users\ayush\OneDrive\Desktop\customer-operations-sla-analytics\Notebook\data\raw
Contents of RAW_DATA_PATH parent: ['raw']
